# ZimTech Retail Holdings: Sales Intelligence Analysis
**Assessment:** INJIVA-DS-DA-PANDAS-001 · **Department:** INJIVA Digital Systems, Data & Analytics

This notebook walks through Stages 1 to 12 of the assessment. The logic lives in the `src/` modules
(validation, cleaning, analytics, pipeline). The notebook **calls** those functions and explains the
results, so no code is copied between cells.

> No charts are used (assessment restriction). Results are shown as DataFrames and written explanations.

**Run order:** use *Kernel → Restart & Run All*. Every cell relies only on cells above it.

## Setup
We add the project root to Python's search path so `import src...` works from inside `notebooks/`. We use a relative path, not a hard-coded one, so this works on any computer.

In [ ]:
import sys
from pathlib import Path

# If the notebook runs from notebooks/, the project root is one level up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd   
import numpy as np   

from src import validation, cleaning, analytics
from src.source_data import sales_data, get_sales_data
from src.pipeline import run_pipeline

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

In [4]:
# Verify list lengths BEFORE building the DataFrame
record_count = validation.check_list_lengths(sales_data)
print(f"All {len(sales_data)} lists have {record_count} values each.")

sales_df = validation.build_sales_dataframe(get_sales_data())
sales_df   # display the complete DataFrame

All 11 lists have 20 values each.


,Transaction_ID,Date,Product,Category,Quantity,Unit_Price,Unit_Cost,Salesperson,City,Payment_Method,Discount_Percent
0,TX001,2026-01-05,Laptop,Computers,2,650.00,500,John,Bulawayo,Cash,0
1,TX002,06/01/2026,Smartphone,Mobile,5,300.00,220,Mary,Harare,EcoCash,5
2,TX003,2026-01-07,keyboard,Accessories,10,25.00,12,Peter,Gweru,Bank Transfer,0
3,TX004,2026-01-08,Monitor,Computers,3,180.00,130,John,Bulawayo,Cash,10
4,TX005,09-01-2026,Printer,Office Equipment,2,220.00,160,Sarah,Mutare,EcoCash,0
5,TX006,2026-01-10,LAPTOP,Computers,1,700.00,520,Mary,Harare,Bank Transfer,5
6,TX007,2026-01-11,Mouse,Accessories,15,15.00,7,Peter,Gweru,Cash,0
7,TX008,12/01/2026,Smartphone,Mobile,4,320.00,240,Sarah,Mutare,EcoCash,10
8,TX009,2026-01-13,monitor,Computers,-2,200.00,145,John,Bulawayo,Cash,0
9,TX010,2026-01-14,Keyboard,Accessories,8,30.00,14,Mary,Harare,Bank Transfer,0


In [4]:
# Verify list lengths BEFORE building the DataFrame
record_count = validation.check_list_lengths(sales_data)
print(f"All {len(sales_data)} lists have {record_count} values each.")

sales_df = validation.build_sales_dataframe(get_sales_data())
sales_df   # display the complete DataFrame

All 11 lists have 20 values each.


,Transaction_ID,Date,Product,Category,Quantity,Unit_Price,Unit_Cost,Salesperson,City,Payment_Method,Discount_Percent
0,TX001,2026-01-05,Laptop,Computers,2,650.00,500,John,Bulawayo,Cash,0
1,TX002,06/01/2026,Smartphone,Mobile,5,300.00,220,Mary,Harare,EcoCash,5
2,TX003,2026-01-07,keyboard,Accessories,10,25.00,12,Peter,Gweru,Bank Transfer,0
3,TX004,2026-01-08,Monitor,Computers,3,180.00,130,John,Bulawayo,Cash,10
4,TX005,09-01-2026,Printer,Office Equipment,2,220.00,160,Sarah,Mutare,EcoCash,0
5,TX006,2026-01-10,LAPTOP,Computers,1,700.00,520,Mary,Harare,Bank Transfer,5
6,TX007,2026-01-11,Mouse,Accessories,15,15.00,7,Peter,Gweru,Cash,0
7,TX008,12/01/2026,Smartphone,Mobile,4,320.00,240,Sarah,Mutare,EcoCash,10
8,TX009,2026-01-13,monitor,Computers,-2,200.00,145,John,Bulawayo,Cash,0
9,TX010,2026-01-14,Keyboard,Accessories,8,30.00,14,Mary,Harare,Bank Transfer,0


---
## Stage 1: DataFrame Construction and Schema Validation
First we check that every list in the dictionary has the same length, then we build `sales_df` from a
**deep copy** of the dictionary, so the original `sales_data` can never be changed.

In [4]:
# Verify list lengths BEFORE building the DataFrame
record_count = validation.check_list_lengths(sales_data)
print(f"All {len(sales_data)} lists have {record_count} values each.")

sales_df = validation.build_sales_dataframe(get_sales_data())
sales_df   # display the complete DataFrame

All 11 lists have 20 values each.


,Transaction_ID,Date,Product,Category,Quantity,Unit_Price,Unit_Cost,Salesperson,City,Payment_Method,Discount_Percent
0,TX001,2026-01-05,Laptop,Computers,2,650.00,500,John,Bulawayo,Cash,0
1,TX002,06/01/2026,Smartphone,Mobile,5,300.00,220,Mary,Harare,EcoCash,5
2,TX003,2026-01-07,keyboard,Accessories,10,25.00,12,Peter,Gweru,Bank Transfer,0
3,TX004,2026-01-08,Monitor,Computers,3,180.00,130,John,Bulawayo,Cash,10
4,TX005,09-01-2026,Printer,Office Equipment,2,220.00,160,Sarah,Mutare,EcoCash,0
5,TX006,2026-01-10,LAPTOP,Computers,1,700.00,520,Mary,Harare,Bank Transfer,5
6,TX007,2026-01-11,Mouse,Accessories,15,15.00,7,Peter,Gweru,Cash,0
7,TX008,12/01/2026,Smartphone,Mobile,4,320.00,240,Sarah,Mutare,EcoCash,10
8,TX009,2026-01-13,monitor,Computers,-2,200.00,145,John,Bulawayo,Cash,0
9,TX010,2026-01-14,Keyboard,Accessories,8,30.00,14,Mary,Harare,Bank Transfer,0


In [5]:
# Shape, column names and data types
print("Rows, columns:", sales_df.shape)
print("Columns:", list(sales_df.columns))
sales_df.dtypes

Rows, columns: (20, 11)
Columns: ['Transaction_ID', 'Date', 'Product', 'Category', 'Quantity', 'Unit_Price', 'Unit_Cost', 'Salesperson', 'City', 'Payment_Method', 'Discount_Percent']


Transaction_ID       object
Date                 object
Product              object
Category             object
Quantity              int64
Unit_Price          float64
Unit_Cost             int64
Salesperson          object
City                 object
Payment_Method       object
Discount_Percent      int64
dtype: object

**Note:** `Unit_Price` is `float64`, not integer, because one value is `None` (stored as NaN). `Date` is text, not a datetime. A DataFrame that loads successfully is not necessarily valid.

In [6]:
sales_df.head()

,Transaction_ID,Date,Product,Category,Quantity,Unit_Price,Unit_Cost,Salesperson,City,Payment_Method,Discount_Percent
0,TX001,2026-01-05,Laptop,Computers,2,650.00,500,John,Bulawayo,Cash,0
1,TX002,06/01/2026,Smartphone,Mobile,5,300.00,220,Mary,Harare,EcoCash,5
2,TX003,2026-01-07,keyboard,Accessories,10,25.00,12,Peter,Gweru,Bank Transfer,0
3,TX004,2026-01-08,Monitor,Computers,3,180.00,130,John,Bulawayo,Cash,10
4,TX005,09-01-2026,Printer,Office Equipment,2,220.00,160,Sarah,Mutare,EcoCash,0


In [7]:
sales_df.tail()

,Transaction_ID,Date,Product,Category,Quantity,Unit_Price,Unit_Cost,Salesperson,City,Payment_Method,Discount_Percent
15,TX016,2026-01-20,Monitor,Computers,4,190.00,140,Sarah,Mutare,EcoCash,0
16,TX017,2026-01-21,Keyboard,Accessories,9,28.00,13,Peter,Gweru,Cash,0
17,TX018,2026-01-22,Printer,Office Equipment,2,230.00,170,Mary,Harare,Bank Transfer,5
18,TX019,2026-01-23,Smartphone,Mobile,6,330.00,250,Sarah,Mutare,EcoCash,10
19,TX020,2026-01-24,Mouse,Accessories,20,15.00,7,Peter,Gweru,Cash,0


In [8]:
sales_df.info()
print()
print(f"Memory usage (deep): {sales_df.memory_usage(deep=True).sum():,} bytes")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction_ID    20 non-null     object 
 1   Date              20 non-null     object 
 2   Product           20 non-null     object 
 3   Category          20 non-null     object 
 4   Quantity          20 non-null     int64  
 5   Unit_Price        19 non-null     float64
 6   Unit_Cost         20 non-null     int64  
 7   Salesperson       20 non-null     object 
 8   City              20 non-null     object 
 9   Payment_Method    20 non-null     object 
 10  Discount_Percent  20 non-null     int64  
dtypes: float64(1), int64(3), object(7)
memory usage: 1.8+ KB

Memory usage (deep): 8,636 bytes


### Mandatory columns, expected types and schema validation

In [9]:
pd.DataFrame({
    "Mandatory": [c in validation.MANDATORY_COLUMNS for c in validation.EXPECTED_RAW_TYPES],
    "Expected_Raw_Type": validation.EXPECTED_RAW_TYPES.values(),
    "Expected_Clean_Type": validation.EXPECTED_CLEAN_TYPES.values(),
    "Actual_dtype": [str(sales_df[c].dtype) for c in validation.EXPECTED_RAW_TYPES],
}, index=validation.EXPECTED_RAW_TYPES.keys())

,Mandatory,Expected_Raw_Type,Expected_Clean_Type,Actual_dtype
Transaction_ID,True,text,text,object
Date,True,text,datetime,object
Product,True,text,text,object
Category,True,text,text,object
Quantity,True,integer,integer,int64
Unit_Price,True,numeric,numeric,float64
Unit_Cost,True,numeric,numeric,int64
Salesperson,True,text,text,object
City,True,text,text,object
Payment_Method,True,text,text,object


In [10]:
# The schema passes for the source data...
print(validation.validate_schema(sales_df))

# ...and gives a MEANINGFUL error when a required column is missing.
# We catch only SchemaValidationError, never a broad "except Exception".
try:
    validation.validate_schema(sales_df.drop(columns=["Unit_Cost"]))
except validation.SchemaValidationError as error:
    print("Caught expected error ->", error)

{'rows': 20, 'columns': 11, 'missing_columns': [], 'type_errors': {}, 'status': 'PASSED'}
Caught expected error -> Missing mandatory column(s): ['Unit_Cost']. Columns found: ['Transaction_ID', 'Date', 'Product', 'Category', 'Quantity', 'Unit_Price', 'Salesperson', 'City', 'Payment_Method', 'Discount_Percent']


---
## Stage 2: Data Quality Investigation
Every rule is applied with Boolean masks. Each finding becomes a row in the Data Quality Report. A rule
that finds nothing still records **"No violations found"**, so the report shows every rule was checked.

In [11]:
validation.missing_value_summary(sales_df)

,Missing_Count,Missing_Percent
Column,,
Transaction_ID,0,0.00
Date,0,0.00
Product,0,0.00
Category,0,0.00
Quantity,0,0.00
Unit_Price,1,5.00
Unit_Cost,0,0.00
Salesperson,0,0.00
City,0,0.00


In [12]:
quality_report = validation.run_data_quality_checks(sales_df)
issues_only = quality_report[quality_report["Transaction_ID"] != "ALL"]
print(f"{len(quality_report)} report rows: {len(issues_only)} specific findings, "
      f"{(quality_report['Resolution_Status'] == 'Passed').sum()} rules passed with no violations.")
quality_report

24 report rows: 11 specific findings, 11 rules passed with no violations.


,Rule_ID,Rule_Name,Transaction_ID,Affected_Column,Original_Value,Issue_Description,Severity,Recommended_Action,Resolution_Status
0,R01,Missing value,ALL,All except Unit_Price,,No violations found,None,No action required,Passed
1,R02,Duplicate Transaction_ID,ALL,Transaction_ID,,No violations found,None,No action required,Passed
2,R03,Fully duplicated record,ALL,All columns,,No violations found,None,No action required,Passed
3,R04,Negative quantity,TX009,Quantity,-2,Negative quantity: possible return/reversal or...,High,Exclude from sales totals; confirm with Sales ...,Rejected - pending manual review
4,R05,Zero quantity,TX015,Quantity,0,Zero units sold: no economic transaction occur...,Medium,Exclude from sales totals; confirm whether the...,Rejected - pending manual review
5,R06,Missing selling price,TX012,Unit_Price,None,Unit_Price is missing; revenue cannot be calcu...,High,Do not impute; obtain actual invoice price fro...,Rejected - pending manual review
6,R07,Invalid unit price,ALL,Unit_Price,,No violations found,None,No action required,Passed
7,R08,Invalid unit cost,ALL,Unit_Cost,,No violations found,None,No action required,Passed
8,R09,Unit price below unit cost,ALL,Unit_Price,,No violations found,None,No action required,Passed
9,R10,Inconsistent product name,TX003,Product,keyboard,Non-standard casing; standard form is 'Keyboard',Low,Standardise to Title Case,Resolved - standardised during cleaning


In [13]:
# Summary by severity
quality_report.groupby("Severity")["Rule_ID"].count().rename("Findings")

Severity
High       2
Low        9
Medium     2
None      11
Name: Findings, dtype: int64

**Interpretation**
- **TX009** (Quantity −2): this *may* be a return, but there is no link to an original sale. We treat it as unconfirmed and do not count it as revenue.
- **TX012** (missing Unit_Price): a missing price is **not** zero. Printer prices elsewhere differ (220 and 230), so imputing one would invent revenue.
- **TX015** (Quantity 0): no goods moved, so this is not a sale.
- Three product names are in the wrong case and five dates are in non-ISO formats. Cleaning can fix these safely.
- No duplicate IDs, no fully duplicated rows, no invalid costs, no suspicious discounts, and no product-category conflicts were found.

---
## Stage 3: Data Cleaning and Standardisation
`clean_sales_data()` works on a copy, keeps the original values in `Original_Product` and `Original_Date`,
records every change in `Cleaning_Notes`, and **separates** rejected records instead of deleting them.
All assumptions (A1 to A7) are written in the docstring of `src/cleaning.py`.

In [14]:
cleaned_df, rejected_df, cleaning_summary = cleaning.clean_sales_data(sales_df)
cleaning_summary

,Metric,Value
0,Records in source data,20
1,Records accepted,17
2,Records rejected,3
3,Product names standardised,3
4,Dates converted from non-ISO format,5
5,Accepted records flagged for manual review,0
6,Duplicate Transaction_IDs after cleaning,0


In [15]:
# Audit trail: rows whose values were changed during cleaning
cleaned_df.loc[cleaned_df["Cleaning_Notes"] != "",
               ["Transaction_ID", "Original_Product", "Product", "Original_Date", "Date", "Cleaning_Notes"]]

Column,Transaction_ID,Original_Product,Product,Original_Date,Date,Cleaning_Notes
1,TX002,Smartphone,Smartphone,06/01/2026,2026-01-06,Date converted from '06/01/2026' (day-first)
2,TX003,keyboard,Keyboard,2026-01-07,2026-01-07,Product standardised from 'keyboard'
4,TX005,Printer,Printer,09-01-2026,2026-01-09,Date converted from '09-01-2026' (day-first)
5,TX006,LAPTOP,Laptop,2026-01-10,2026-01-10,Product standardised from 'LAPTOP'
7,TX008,Smartphone,Smartphone,12/01/2026,2026-01-12,Date converted from '12/01/2026' (day-first)


In [16]:
rejected_df[['Transaction_ID', 'Product', 'Quantity', 'Unit_Price', 'Rejection_Reason']]

Column,Transaction_ID,Product,Quantity,Unit_Price,Rejection_Reason
0,TX009,Monitor,-2,200.00,Negative Quantity (possible return - no origin...
1,TX012,Printer,2,NaN,Missing Unit_Price (not imputed - needs actual...
2,TX015,Laptop,0,670.00,Zero Quantity (no goods sold)


In [17]:
# Checks: no duplicates introduced, cleaned data valid, original untouched
cleaning.verify_no_new_duplicates(sales_df, cleaned_df)
validation.validate_cleaned_data(cleaned_df)
print("Cleaned dtypes:\n", cleaned_df[["Date", "Quantity", "Unit_Price"]].dtypes, sep="")
print("Original sales_df still has the raw product 'LAPTOP':", "LAPTOP" in sales_df["Product"].values)

Cleaned dtypes:
Column
Date          datetime64[ns]
Quantity               int64
Unit_Price           float64
dtype: object
Original sales_df still has the raw product 'LAPTOP': True


---
## Stage 4: Financial Data Transformation
All formulas are vectorised. Discounts are divided by 100, and profit is calculated **after** the discount.
Margins use `safe_margin()`, which returns NaN instead of dividing by zero.

In [18]:
fin_df = analytics.add_financial_columns(cleaned_df)
fin_df[["Transaction_ID", "Product", "Quantity", "Unit_Price", "Discount_Percent", "Gross_Revenue",
        "Discount_Amount", "Net_Revenue", "Total_Cost", "Gross_Profit", "Gross_Profit_Margin"]]

Column,Transaction_ID,Product,Quantity,Unit_Price,Discount_Percent,Gross_Revenue,Discount_Amount,Net_Revenue,Total_Cost,Gross_Profit,Gross_Profit_Margin
0,TX001,Laptop,2,650.00,0.00,"1,300.00",0.00,"1,300.00","1,000.00",300.00,23.08
1,TX002,Smartphone,5,300.00,5.00,"1,500.00",75.00,"1,425.00","1,100.00",325.00,22.81
2,TX003,Keyboard,10,25.00,0.00,250.00,0.00,250.00,120.00,130.00,52.00
3,TX004,Monitor,3,180.00,10.00,540.00,54.00,486.00,390.00,96.00,19.75
4,TX005,Printer,2,220.00,0.00,440.00,0.00,440.00,320.00,120.00,27.27
5,TX006,Laptop,1,700.00,5.00,700.00,35.00,665.00,520.00,145.00,21.80
6,TX007,Mouse,15,15.00,0.00,225.00,0.00,225.00,105.00,120.00,53.33
7,TX008,Smartphone,4,320.00,10.00,"1,280.00",128.00,"1,152.00",960.00,192.00,16.67
8,TX010,Keyboard,8,30.00,0.00,240.00,0.00,240.00,112.00,128.00,53.33
9,TX011,Laptop,3,680.00,5.00,"2,040.00",102.00,"1,938.00","1,530.00",408.00,21.05


In [19]:
# Independent verification: net revenue recalculated as Q * P * (1 - d/100)
print("Financial calculations verified:", analytics.verify_financial_calculations(fin_df))
print("Division-by-zero check:", analytics.safe_margin(50, 0))
analytics.financial_summary(fin_df)

Financial calculations verified: True
Division-by-zero check: nan


,Metric,Value
0,Valid transactions,17
1,Total units sold,111
2,Total gross revenue (USD),"14,033.00"
3,Total discount amount (USD),692.50
4,Total net revenue (USD),"13,340.50"
5,Total cost (USD),"10,085.00"
6,Total gross profit (USD),"3,255.50"
7,Overall gross profit margin (%),24.40
8,Average transaction value (USD),784.74
9,Loss-making transactions,0


In [20]:
# Overall margin: correct method vs the common mistake
correct = fin_df["Gross_Profit"].sum() / fin_df["Net_Revenue"].sum() * 100
wrong = fin_df["Gross_Profit_Margin"].mean()
print(f"Correct overall margin (total profit / total revenue): {correct:.2f}%")
print(f"Wrong method (average of row margins):                {wrong:.2f}%")
print("Loss-making transactions:", len(analytics.loss_making_transactions(fin_df)))

Correct overall margin (total profit / total revenue): 24.40%
Wrong method (average of row margins):                32.83%
Loss-making transactions: 0


The 'wrong' average is inflated because small, high-margin accessory sales count as much as large laptop sales. No transaction made a loss: every selling price (after discount) is above unit cost.

---
## Stage 5: Advanced Sales Performance Analysis
**Metric definitions:** *revenue* = Net_Revenue (after discount); *profit* = Gross_Profit; *volume* = units;
*transactions* = count of valid rows. **Ties** are ranked with `method="min"`.

In [21]:
fs = analytics.financial_summary(fin_df).set_index("Metric")["Value"]
print(f"Total net revenue:       ${fs['Total net revenue (USD)']:,.2f}")
print(f"Valid transactions:      {fs['Valid transactions']}")
print(f"Total units sold:        {fs['Total units sold']}")
print(f"Average transaction:     ${fs['Average transaction value (USD)']:,.2f}")
analytics.top_transactions(fin_df, 5)   # uses nlargest()

Total net revenue:       $13,340.50
Valid transactions:      17
Total units sold:        111
Average transaction:     $784.74


Column,Transaction_ID,Date,Product,City,Salesperson,Quantity,Net_Revenue,Gross_Profit,Gross_Profit_Margin
9,TX011,2026-01-15,Laptop,Bulawayo,John,3,"1,938.00",408.00,21.05
15,TX019,2026-01-23,Smartphone,Mutare,Sarah,6,"1,782.00",282.00,15.82
11,TX014,2026-01-18,Smartphone,Harare,Mary,5,"1,472.50",297.50,20.20
1,TX002,2026-01-06,Smartphone,Harare,Mary,5,"1,425.00",325.00,22.81
0,TX001,2026-01-05,Laptop,Bulawayo,John,2,"1,300.00",300.00,23.08


In [22]:
products = analytics.product_performance(fin_df)   # groupby + agg + rank
products[["Transactions", "Units_Sold", "Net_Revenue", "Gross_Profit", "Gross_Profit_Margin",
          "Revenue_Rank", "Profit_Rank", "Below_Average_Margin"]]

,Transactions,Units_Sold,Net_Revenue,Gross_Profit,Gross_Profit_Margin,Revenue_Rank,Profit_Rank,Below_Average_Margin
Product,,,,,,,,
Smartphone,4,20,"5,831.50","1,096.50",18.80,1,1,True
Laptop,3,6,"3,903.00",853.00,21.85,2,2,True
Monitor,2,7,"1,246.00",296.00,23.76,3,5,True
Printer,2,4,877.00,217.00,24.74,4,6,False
Keyboard,3,27,742.00,393.00,52.96,5,4,False
Mouse,3,47,741.00,400.00,53.98,6,3,False


In [23]:
categories = analytics.category_performance(fin_df)
categories.sort_values("Units_Sold", ascending=False)[["Units_Sold", "Net_Revenue", "Gross_Profit", "Avg_Discount_Percent"]]

,Units_Sold,Net_Revenue,Gross_Profit,Avg_Discount_Percent
Category,,,,
Accessories,74,"1,483.00",793.00,0.00
Mobile,20,"5,831.50","1,096.50",7.50
Computers,13,"5,149.00","1,149.00",4.00
Office Equipment,4,877.00,217.00,2.50


In [24]:
cities = analytics.city_performance(fin_df)
cities[["Transactions", "Net_Revenue", "Gross_Profit", "Gross_Profit_Margin", "Revenue_Share_Percent", "Data_Status"]]

,Transactions,Net_Revenue,Gross_Profit,Gross_Profit_Margin,Revenue_Share_Percent,Data_Status
City,,,,,,
Harare,5,"4,239.50",992.50,23.41,31.78,Records present
Mutare,4,"4,134.00",794.00,19.21,30.99,Records present
Bulawayo,3,"3,724.00",804.00,21.59,27.91,Records present
Gweru,5,"1,243.00",665.00,53.50,9.32,Records present
Masvingo,<NA>,NaN,NaN,NaN,NaN,No records in sample - activity unknown


Masvingo is shown as **unknown (NaN)**, not zero: the sample has no Masvingo records, which does not prove there were no sales.

In [25]:
people = analytics.salesperson_performance(fin_df)
people[["Net_Revenue", "Gross_Profit", "Gross_Profit_Margin", "Revenue_vs_Average", "Profit_vs_Average"]]

,Net_Revenue,Gross_Profit,Gross_Profit_Margin,Revenue_vs_Average,Profit_vs_Average
Salesperson,,,,,
Mary,"4,239.50",992.50,23.41,At/above average,At/above average
Sarah,"4,134.00",794.00,19.21,At/above average,Below average
John,"3,724.00",804.00,21.59,At/above average,Below average
Peter,"1,243.00",665.00,53.50,Below average,Below average


In [26]:
analytics.payment_method_summary(fin_df)

,Transactions,Net_Revenue,Usage_Percent
Payment_Method,,,
Cash,7,"4,717.00",41.18
Bank Transfer,5,"3,064.50",29.41
EcoCash,5,"5,559.00",29.41


In [27]:
# transform(): adds each transaction's share of its city's revenue to EVERY row
analytics.add_group_shares(fin_df)[["Transaction_ID", "City", "Net_Revenue", "City_Total_Revenue",
                                     "Share_of_City_Revenue_Percent"]].sort_values("City")

Column,Transaction_ID,City,Net_Revenue,City_Total_Revenue,Share_of_City_Revenue_Percent
0,TX001,Bulawayo,"1,300.00","3,724.00",34.91
3,TX004,Bulawayo,486.00,"3,724.00",13.05
9,TX011,Bulawayo,"1,938.00","3,724.00",52.04
13,TX017,Gweru,252.00,"1,243.00",20.27
10,TX013,Gweru,216.00,"1,243.00",17.38
6,TX007,Gweru,225.00,"1,243.00",18.10
16,TX020,Gweru,300.00,"1,243.00",24.14
2,TX003,Gweru,250.00,"1,243.00",20.11
11,TX014,Harare,"1,472.50","4,239.50",34.73
1,TX002,Harare,"1,425.00","4,239.50",33.61


In [28]:
# query() and Boolean indexing
print("Products with below-average margin:", list(products.index[products["Below_Average_Margin"]]))
print("Salespeople below average revenue:", list(people.index[people["Revenue_vs_Average"] == "Below average"]))
high_value = fin_df[(fin_df["Net_Revenue"] > 1000) & (fin_df["Discount_Percent"] > 0)]  # Boolean indexing
print("Discounted transactions over $1,000:", list(high_value["Transaction_ID"]))

Products with below-average margin: ['Smartphone', 'Laptop', 'Monitor']
Salespeople below average revenue: ['Peter']
Discounted transactions over $1,000: ['TX002', 'TX008', 'TX011', 'TX014', 'TX019']


In [29]:
# Reconciliation: grouped totals must equal the transaction total
for name, table in [("products", products), ("cities", cities), ("salespeople", people)]:
    print(name, "reconciled:", analytics.reconcile_totals(table, fin_df))

products reconciled: True
cities reconciled: True
salespeople reconciled: True


**Important observation:** each salesperson works in exactly one city (Mary = Harare, Sarah = Mutare,
John = Bulawayo, Peter = Gweru). City results and salesperson results are therefore the same numbers, and
this data cannot separate a *person* effect from a *market* effect.

In [30]:
fin_df.groupby('Salesperson')['City'].unique()

Salesperson
John     [Bulawayo]
Mary       [Harare]
Peter       [Gweru]
Sarah      [Mutare]
Name: City, dtype: object

---
## Stage 6: Time-Series Sales Analysis
Dates were converted to datetime in Stage 3, so sorting is chronological, not alphabetical.
The daily table covers **every day of January 2026**. Days without valid sales get NaN (unknown), never 0.

In [31]:
dated = analytics.add_date_parts(fin_df)
dated[["Transaction_ID", "Date", "Day", "Month", "Year", "Weekday"]].head()

Column,Transaction_ID,Date,Day,Month,Year,Weekday
0,TX001,2026-01-05,5,1,2026,Monday
1,TX002,2026-01-06,6,1,2026,Tuesday
2,TX003,2026-01-07,7,1,2026,Wednesday
3,TX004,2026-01-08,8,1,2026,Thursday
4,TX005,2026-01-09,9,1,2026,Friday


In [32]:
raw_dates = sales_df["Date"].apply(validation.parse_date_value)
daily = analytics.daily_revenue(fin_df, raw_dates=raw_dates)
daily

,Net_Revenue,Gross_Profit,Transaction_Count,Rolling_3_Transaction_Days,Cumulative_Revenue,Rolling_3_Calendar_Days,Day_Status
Date,,,,,,,
2026-01-01,NaN,NaN,0,NaN,NaN,NaN,No records
2026-01-02,NaN,NaN,0,NaN,NaN,NaN,No records
2026-01-03,NaN,NaN,0,NaN,NaN,NaN,No records
2026-01-04,NaN,NaN,0,NaN,NaN,NaN,No records
2026-01-05,"1,300.00",300.00,1,NaN,"1,300.00","1,300.00",Valid sales
2026-01-06,"1,425.00",325.00,1,NaN,"2,725.00","1,362.50",Valid sales
2026-01-07,250.00,130.00,1,991.67,"2,975.00",991.67,Valid sales
2026-01-08,486.00,96.00,1,720.33,"3,461.00",720.33,Valid sales
2026-01-09,440.00,120.00,1,392.00,"3,901.00",392.00,Valid sales


In [33]:
valid_days = daily.dropna(subset=["Net_Revenue"])
print("Highest-revenue date:       ", valid_days["Net_Revenue"].idxmax().date(), f"${valid_days['Net_Revenue'].max():,.2f}")
positive = valid_days[valid_days["Net_Revenue"] > 0]
print("Lowest positive-revenue date:", positive["Net_Revenue"].idxmin().date(), f"${positive['Net_Revenue'].min():,.2f}")
print()
print(daily["Day_Status"].value_counts())
print("\nDays with only rejected records:", [d.date().isoformat() for d in daily.index[daily['Day_Status'] == 'Only rejected records']])

Highest-revenue date:        2026-01-15 $1,938.00
Lowest positive-revenue date: 2026-01-17 $216.00

Day_Status
Valid sales              17
No records               11
Only rejected records     3
Name: count, dtype: int64

Days with only rejected records: ['2026-01-13', '2026-01-16', '2026-01-19']


In [34]:
analytics.compare_periods(fin_df)

,Period,Calendar_Days,Days_With_Valid_Sales,Transactions,Net_Revenue,Gross_Profit,Revenue_per_Calendar_Day,Change_vs_Previous_Percent
0,2026-01-05 to 2026-01-14,10,9,9,"6,183.00","1,556.00",618.30,NaN
1,2026-01-15 to 2026-01-24,10,8,8,"7,157.50","1,699.50",715.75,15.76


**Effect of missing dates**
- Records exist only for 5 to 24 January. 1 to 4 and 25 to 31 January have *no records*. That is a gap in the sample, not proof of zero sales.
- 9, 16 and 19 January had transactions, but all of them were rejected. Their revenue is unknown, not zero.
- **Transaction-day** rolling averages skip gaps, so a window can cover more than three calendar days. **Calendar-day** rolling averages use a true 3-day window and average only the days that have data. Filling gaps with 0 would wrongly pull both averages down.
- Both periods compared are 10 calendar days long, so comparing their totals is fair. Revenue per day is also shown.

---
## Stage 7: Product and Sales Segmentation
**Threshold justification:** transaction value segments use the **tertiles** (33rd and 67th percentiles)
of net revenue. This gives roughly equal groups based on the data rather than an arbitrary number.
The conditions do not overlap, and `np.select` has a default, so every transaction gets a label.

In [35]:
segmented, thresholds = analytics.classify_transaction_value(fin_df)
print({k: round(v, 2) for k, v in thresholds.items()})
print(segmented["Value_Segment"].value_counts(), "\nUnclassified:", segmented["Value_Segment"].isna().sum())
segmented[["Transaction_ID", "Product", "Net_Revenue", "Value_Segment"]].sort_values("Net_Revenue", ascending=False)

{'low_upper_bound': 345.67, 'medium_upper_bound': 1021.33}
Value_Segment
High      6
Low       6
Medium    5
Name: count, dtype: int64 
Unclassified: 0


Column,Transaction_ID,Product,Net_Revenue,Value_Segment
9,TX011,Laptop,"1,938.00",High
15,TX019,Smartphone,"1,782.00",High
11,TX014,Smartphone,"1,472.50",High
1,TX002,Smartphone,"1,425.00",High
0,TX001,Laptop,"1,300.00",High
7,TX008,Smartphone,"1,152.00",High
12,TX016,Monitor,760.00,Medium
5,TX006,Laptop,665.00,Medium
3,TX004,Monitor,486.00,Medium
4,TX005,Printer,440.00,Medium


In [36]:
analytics.segment_products(fin_df)[["Net_Revenue", "Revenue_Share_Percent", "Gross_Profit_Margin", "Segment", "Low_Margin"]]

,Net_Revenue,Revenue_Share_Percent,Gross_Profit_Margin,Segment,Low_Margin
Product,,,,,
Smartphone,"5,831.50",43.71,18.80,Revenue driver (low margin),True
Laptop,"3,903.00",29.26,21.85,Revenue driver (low margin),True
Monitor,"1,246.00",9.34,23.76,Underperformer,True
Printer,877.00,6.57,24.74,Margin niche (low revenue),False
Keyboard,742.00,5.56,52.96,Margin niche (low revenue),False
Mouse,741.00,5.55,53.98,Margin niche (low revenue),False


In [37]:
analytics.segment_categories(fin_df)[["Net_Revenue", "Revenue_Share_Percent", "Gross_Profit_Margin", "Category_Class"]]

,Net_Revenue,Revenue_Share_Percent,Gross_Profit_Margin,Category_Class
Category,,,,
Mobile,"5,831.50",43.71,18.80,"Core - revenue driver, low margin"
Computers,"5,149.00",38.60,22.32,"Core - revenue driver, low margin"
Accessories,"1,483.00",11.12,53.47,Niche - high margin
Office Equipment,877.00,6.57,24.74,Niche - high margin


In [38]:
high_disc, disc_threshold = analytics.high_discount_transactions(fin_df)
print(f"Unusually high discount = above the 75th percentile ({disc_threshold:.0f}%)")
high_disc[["Transaction_ID", "Product", "Discount_Percent", "Discount_Amount", "Gross_Profit_Margin"]]

Unusually high discount = above the 75th percentile (5%)


Column,Transaction_ID,Product,Discount_Percent,Discount_Amount,Gross_Profit_Margin
3,TX004,Monitor,10.00,54.00,19.75
7,TX008,Smartphone,10.00,128.00,16.67
15,TX019,Smartphone,10.00,198.00,15.82


In [39]:
people[['Net_Revenue', 'Revenue_Share_Percent']]   # salesperson revenue contribution

,Net_Revenue,Revenue_Share_Percent
Salesperson,,
Mary,"4,239.50",31.78
Sarah,"4,134.00",30.99
John,"3,724.00",27.91
Peter,"1,243.00",9.32


**Limitations of revenue-based segmentation:** Keyboards and mice rank last on revenue but have margins
above 50 %. Low revenue does not mean low profitability. With only 17 transactions, one large order can
move a product between segments, so these labels are indicative only.

---
## Stage 8: Multi-DataFrame Operations
The reference table has one row per product (the "one" side), and sales have many rows per product.
The merge uses `validate="many_to_one"`, so pandas raises an error if the reference contains duplicate keys.

In [40]:
reference_df = analytics.build_reference_dataframe()   # validates codes and standardises names
reference_df

,Product_Code,Product_Name,Product_Category,Reorder_Threshold
0,P001,Laptop,Computers,5
1,P002,Smartphone,Mobile,10
2,P003,Keyboard,Accessories,20
3,P004,Monitor,Computers,5
4,P005,Printer,Office Equipment,3
5,P006,Mouse,Accessories,25
6,P007,Tablet,Mobile,8


In [41]:
merge = analytics.merge_report(fin_df, reference_df)
merge["checks"]

,Check,Value
0,Rows before merge,17.00
1,Rows after left join,17.00
2,Rows after inner join,17.00
3,Revenue before merge,"13,340.50"
4,Revenue after left join,"13,340.50"
5,Duplicate Transaction_IDs after merge,0.00
6,Unmatched sales rows,0.00
7,Reference products with no sales,1.00
8,Category mismatches,0.00


In [42]:
inner = merge["inner"]; left = merge["left"]
print("Inner join rows:", len(inner), "| Left join rows:", len(left))
print("Unmatched sales rows:", len(merge["unmatched_sales"]))
print("Reference products with no sales:\n", merge["unused_reference"])
left[["Transaction_ID", "Product", "Product_Code", "Reorder_Threshold", "_merge"]].head()

Inner join rows: 17 | Left join rows: 17
Unmatched sales rows: 0
Reference products with no sales:
    Product_Code Product_Name
17         P007       Tablet


,Transaction_ID,Product,Product_Code,Reorder_Threshold,_merge
0,TX001,Laptop,P001,5,both
1,TX002,Smartphone,P002,10,both
2,TX003,Keyboard,P003,20,both
3,TX004,Monitor,P004,5,both
4,TX005,Printer,P005,3,both


In [43]:
analytics.city_category_pivot(fin_df)

Category,Accessories,Computers,Mobile,Office Equipment,Total
City,,,,,
Bulawayo,0.00,"3,724.00",0.00,0.00,"3,724.00"
Gweru,"1,243.00",0.00,0.00,0.00,"1,243.00"
Harare,240.00,665.00,"2,897.50",437.00,"4,239.50"
Mutare,0.00,760.00,"2,934.00",440.00,"4,134.00"
Total,"1,483.00","5,149.00","5,831.50",877.00,"13,340.50"


Inner and left joins give the same 17 rows because every sale matches a product. Tablet (P007) appears only in an outer join: it is in the catalogue but has no sales in the sample. Revenue before and after the merge is identical, so no transactions were duplicated.

---
## Stage 9: Advanced Data Manipulation and Business Intelligence

In [44]:
analytics.city_category_aggregation(fin_df)   # MultiIndex: City > Category

Transactions  Net_Revenue  Gross_Profit  Gross_Profit_Margin
City     Category                                                                      
Bulawayo Computers                    3     3,724.00        804.00                21.59
Gweru    Accessories                  5     1,243.00        665.00                53.50
Harare   Accessories                  1       240.00        128.00                53.33
         Computers                    1       665.00        145.00                21.80
         Mobile                       2     2,897.50        622.50                21.48
         Office Equipment             1       437.00         97.00                22.20
Mutare   Computers                    1       760.00        200.00                26.32
         Mobile                       2     2,934.00        474.00                16.16
         Office Equipment             1       440.00        120.00                27.27

In [45]:
# Above-average revenue but below-average margin
analytics.above_revenue_below_margin(fin_df)[["Transaction_ID", "Product", "Net_Revenue", "Gross_Profit_Margin"]]

Column,Transaction_ID,Product,Net_Revenue,Gross_Profit_Margin
0,TX001,Laptop,"1,300.00",23.08
1,TX002,Smartphone,"1,425.00",22.81
7,TX008,Smartphone,"1,152.00",16.67
9,TX011,Laptop,"1,938.00",21.05
11,TX014,Smartphone,"1,472.50",20.20
15,TX019,Smartphone,"1,782.00",15.82


In [46]:
analytics.rank_salespeople_within_city(fin_df)

,City,Salesperson,Net_Revenue,Gross_Profit,Transactions,Rank_in_City
0,Bulawayo,John,"3,724.00",804.00,3,1
1,Gweru,Peter,"1,243.00",665.00,5,1
2,Harare,Mary,"4,239.50",992.50,5,1
3,Mutare,Sarah,"4,134.00",794.00,4,1


In [47]:
analytics.classify_salesperson_performance(fin_df)[['Net_Revenue', 'Gross_Profit', 'Performance_Class']]

,Net_Revenue,Gross_Profit,Performance_Class
Salesperson,,,
Mary,"4,239.50",992.50,Strong
Sarah,"4,134.00",794.00,Mixed
John,"3,724.00",804.00,Mixed
Peter,"1,243.00",665.00,Develop


In [48]:
analytics.cumulative_revenue_by_city(fin_df)

Column,City,Date,Transaction_ID,Net_Revenue,City_Cumulative_Revenue
0,Bulawayo,2026-01-05,TX001,"1,300.00","1,300.00"
3,Bulawayo,2026-01-08,TX004,486.00,"1,786.00"
9,Bulawayo,2026-01-15,TX011,"1,938.00","3,724.00"
2,Gweru,2026-01-07,TX003,250.00,250.00
6,Gweru,2026-01-11,TX007,225.00,475.00
10,Gweru,2026-01-17,TX013,216.00,691.00
13,Gweru,2026-01-21,TX017,252.00,943.00
16,Gweru,2026-01-24,TX020,300.00,"1,243.00"
1,Harare,2026-01-06,TX002,"1,425.00","1,425.00"
5,Harare,2026-01-10,TX006,665.00,"2,090.00"


In [49]:
analytics.product_share_within_category(fin_df)

,Category,Product,Net_Revenue,Category_Revenue,Share_of_Category_Percent
0,Accessories,Keyboard,742.00,"1,483.00",50.03
1,Accessories,Mouse,741.00,"1,483.00",49.97
2,Computers,Laptop,"3,903.00","5,149.00",75.80
3,Computers,Monitor,"1,246.00","5,149.00",24.20
4,Mobile,Smartphone,"5,831.50","5,831.50",100.00
5,Office Equipment,Printer,877.00,877.00,100.00


In [50]:
analytics.top_profitable_categories(fin_df, 3)

,Net_Revenue,Gross_Profit,Gross_Profit_Margin
Category,,,
Computers,"5,149.00","1,149.00",22.32
Mobile,"5,831.50","1,096.50",18.80
Accessories,"1,483.00",793.00,53.47


In [51]:
city_cat = analytics.city_category_aggregation(fin_df)
print("Multi-level total reconciles:", analytics.reconcile_totals(city_cat, fin_df))
analytics.compare_original_and_processed(sales_df, fin_df, rejected_df)

Multi-level total reconciles: True


,Measure,Original,Processed,Rejected
0,Records,20.00,17.00,3
1,Distinct product spellings,9.00,6.00,
2,Distinct date formats/strings,20.00,17.00,
3,Missing Unit_Price values,1.00,0.00,1
4,Non-positive quantities,2.00,0.00,2
5,Gross revenue Q x P (USD),"13,633.00","14,033.00",-400.00


In [52]:
analytics.rejected_financial_impact(rejected_df)

Column,Transaction_ID,Product,Quantity,Unit_Price,Unit_Cost,Discount_Percent,Rejection_Reason,Potential_Net_Revenue
0,TX009,Monitor,-2,200.00,145.00,0.00,Negative Quantity (possible return - no origin...,-400.00
1,TX012,Printer,2,NaN,165.00,0.00,Missing Unit_Price (not imputed - needs actual...,NaN
2,TX015,Laptop,0,670.00,505.00,10.00,Zero Quantity (no goods sold),0.00


**agg() vs transform():** `agg()` reduces each group to **one row** (for example, one row per city).
`transform()` returns a value for **every original row** with the same index, so it can be stored as a new column
(for example, city total next to each transaction).

---
## Stage 10 & 11: Tests and the Reusable Pipeline
The tests live in `tests/test_sales_pipeline.py` and run with `python -m pytest -v`.
The cell below runs the full pipeline from scratch, without using any variables from earlier cells,
which shows it is reproducible.

In [53]:
results = run_pipeline()   # builds, validates, cleans, transforms, aggregates, exports
for path in results["files"]:
    print("written:", path.relative_to(PROJECT_ROOT))
results["sales_summary"]

written: data/cleaned_sales.csv
written: data/rejected_records.csv
written: data/data_quality_report.csv
written: data/sales_summary.csv
written: data/city_performance.csv
written: data/product_performance.csv
written: data/salesperson_performance.csv


,Metric,Value
0,Valid transactions,17
1,Total units sold,111
2,Total gross revenue (USD),"14,033.00"
3,Total discount amount (USD),692.50
4,Total net revenue (USD),"13,340.50"
5,Total cost (USD),"10,085.00"
6,Total gross profit (USD),"3,255.50"
7,Overall gross profit margin (%),24.40
8,Average transaction value (USD),784.74
9,Loss-making transactions,0


In [54]:
# Run the automated tests from inside the notebook
import subprocess
out = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_ROOT, capture_output=True, text=True)
print(out.stdout[-600:])

.....................                                                    [100%]
21 passed in 0.65s



---
## Stage 12: Findings in Brief
The full management report is in `reports/management_report.md`. Every figure in it matches the CSV files in `data/`.

- 17 of 20 records are valid. **Net revenue: $13,340.50**, **gross profit: $3,255.50**, **overall margin: 24.40 %**.
- Smartphones bring in the most revenue (43.7 %) but have the lowest margin (18.8 %). Accessories have margins above 50 %.
- Harare/Mary lead on revenue and profit. Gweru/Peter have the lowest revenue but the highest margin (53.5 %).
- 3 rejected records ($0 included in totals) need Finance and Sales follow-up.